# 03 — Claim Lifecycle Tracking

**Purpose:** match candidate claims one-to-one between every adjacent pair of decoded NCM hierarchy levels and record introductions, removals, retention and factuality-label changes.

The matcher uses TF-IDF cosine similarity followed by Hungarian assignment with a predeclared threshold. It is a lexical correspondence heuristic: an unmatched claim is an observed textual difference, not automatically a hallucination; a label change is not automatically a true correction.

In [ ]:
from pathlib import Path
import sys
import subprocess
import importlib

# Paths
PROJECT_PARENT = Path("/content/drive/MyDrive/NCM Research")
PROJECT_ROOT = PROJECT_PARENT / "github"
SRC_ROOT = PROJECT_ROOT / "src"
CONFIG_PATH = PROJECT_ROOT / "configs" / "experiment.yaml"

# Mount Drive if needed
if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

# Validate the existing project
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(f"Repository not found: {PROJECT_ROOT}")

if not (PROJECT_ROOT / "pyproject.toml").is_file():
    raise FileNotFoundError(
        f"pyproject.toml is missing from {PROJECT_ROOT}"
    )

if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f"Experiment config not found: {CONFIG_PATH}")

# Register the existing local package in this Python environment
subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--no-deps", "--no-build-isolation",
        "-e", str(PROJECT_ROOT),
    ],
    check=True,
)

sys.path.insert(0, str(SRC_ROOT))
importlib.invalidate_caches()

# Import lifecycle pipeline
from ncm_factuality.config import load_config, resolve_paths
from ncm_factuality.lifecycle import run_claim_lifecycle

# Load experiment configuration
cfg = load_config(CONFIG_PATH)
paths = resolve_paths(PROJECT_ROOT, cfg)

# Reuse the existing experiment run directory
run_dir = paths["run_dir"]
print(f"Project: {PROJECT_ROOT}")
print(f"Run directory: {run_dir}")
print("Starting claim lifecycle analysis...")

lifecycle_manifest = run_claim_lifecycle(cfg, run_dir)

print("Claim lifecycle stage completed.")
lifecycle_manifest

## Output contract

`claim_transitions.jsonl` and `lifecycle_manifest.json` describe adjacent-level transitions. These outputs are cheap to rebuild from `claims_annotated.jsonl`; no inference or NLI scoring is repeated.